In [1]:
!pip install earthengine-api geemap -q

In [2]:
import ee


ee.Authenticate()

ee.Initialize(project='crop-mapping-usa-2025')
print('Earth Engine initialised')


Earth Engine initialised


In [8]:
import ee

YEAR      = 2021
STEP_DAYS = 10
N_STEPS   = 36
N_SAMPLES = 2000
TILE_SCALE = 8


BAND_MAP = {
    'B2':  'Blue',
    'B3':  'Green',
    'B4':  'Red',
    'B5':  'RE1',
    'B6':  'RE2',
    'B7':  'RE3',
    'B8A': 'RE4',
    'B8':  'NIR',
    'B11': 'SWIR1',
    'B12': 'SWIR2',
}
S2_BANDS  = list(BAND_MAP.keys())
OUT_BANDS = list(BAND_MAP.values())


CDL_CONFIG = {
    'Arkansas': {
        'from': [1, 2, 3, 5],
        'to':   [1, 2, 3, 4],
        'n_classes': 5,
    },
    'California': {
        'from': [1, 2, 3, 5, 75],
        'to':   [1, 2, 3, 4,  5],
        'n_classes': 6,
    },
}

print('Configuration loaded')
print(f'Bands: {OUT_BANDS}')
print(f'Time steps: t01 – t{N_STEPS:02d}')


Configuration loaded
Bands: ['Blue', 'Green', 'Red', 'RE1', 'RE2', 'RE3', 'RE4', 'NIR', 'SWIR1', 'SWIR2']
Time steps: t01 – t36


In [ ]:
import ee

def mask_s2_clouds(image):
    scl  = image.select('SCL')
    mask = (scl.neq(3)
              .And(scl.neq(8))
              .And(scl.neq(9))
              .And(scl.neq(10)))
    return image.updateMask(mask).divide(10000).select(S2_BANDS)


def make_composite(step_index, roi):
    start_date = ee.Date(f'{YEAR}-01-01')
    t0     = start_date.advance(step_index * STEP_DAYS, 'day')
    t1     = t0.advance(STEP_DAYS, 'day')
    step   = f't{step_index + 1:02d}'
    suffix = f'_{step}'

    s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
            .filterDate(t0, t1)
            .filterBounds(roi)
            .select(S2_BANDS + ['SCL'])
            .map(mask_s2_clouds))

    composite = s2.median().select(S2_BANDS)
    renamed   = composite.rename([b + suffix for b in OUT_BANDS])
    return renamed


def build_stack(roi):
    print('Building composite stack (36 steps)...')
    stack = make_composite(0, roi)
    for i in range(1, N_STEPS):
        stack = stack.addBands(make_composite(i, roi))
        if (i + 1) % 6 == 0:
            print(f'  Added step {i+1}/{N_STEPS}')
    print('Stack built   ')
    return stack


def get_roi(state_name):
    states = ee.FeatureCollection('TIGER/2018/States')
    return (states
            .filter(ee.Filter.eq('NAME', state_name))
            .geometry()
            .simplify(1000))


def get_label_image(state_name):
    cfg  = CDL_CONFIG[state_name]
    cdl  = (ee.ImageCollection('USDA/NASS/CDL')
              .filter(ee.Filter.calendarRange(YEAR, YEAR, 'year'))
              .first()
              .select('cropland'))
    return cdl.remap(cfg['from'], cfg['to'], defaultValue=0).rename('label')


print('Helper functions defined')


Helper functions defined   


In [12]:
import ee

def export_state(state_name, dry_run=False):
    print(f'\n{"="*55}')
    print(f'Processing: {state_name}')
    print('='*55)


    roi = get_roi(state_name)
    print(f'ROI loaded   ')


    label = get_label_image(state_name)
    print(f'CDL labels loaded   ')


    stack = build_stack(roi)


    features = stack.addBands(label)


    cfg     = CDL_CONFIG[state_name]
    samples = features.stratifiedSample(
        numPoints  = N_SAMPLES,
        classBand  = 'label',
        region     = roi,
        scale      = 10,
        tileScale  = TILE_SCALE,
        seed       = 42,
        geometries = True,
    )
    print(f'Sampling configured ({N_SAMPLES} per class)')

    if dry_run:
        print('Dry run — skipping export.')
        return


    task = ee.batch.Export.table.toDrive(
        collection  = samples,
        description = f'MCTNet_{state_name}_{YEAR}',
        fileFormat  = 'CSV',
        folder      = 'MCTNet_Data',
    )
    task.start()
    print(f'Export task started')
    print(f'→ Check https://code.earthengine.google.com/ Tasks tab')
    print(f'→ Output: Google Drive / MCTNet_Data / MCTNet_{state_name}_{YEAR}.csv')
    return task


print('Export function defined')


Export function defined


In [13]:

task_ar = export_state('Arkansas')



Processing: Arkansas
ROI loaded   
CDL labels loaded   
Building composite stack (36 steps)...
  Added step 6/36
  Added step 12/36
  Added step 18/36
  Added step 24/36
  Added step 30/36
  Added step 36/36
Stack built   
Sampling configured (2000 per class)
Export task started
→ Check https://code.earthengine.google.com/ Tasks tab
→ Output: Google Drive / MCTNet_Data / MCTNet_Arkansas_2021.csv


In [ ]:

task_ca = export_state('California')


In [ ]:
import time

def check_tasks():
    tasks = ee.batch.Task.list()
    mctnet_tasks = [t for t in tasks if 'MCTNet' in t.config.get('description', '')]
    if not mctnet_tasks:
        print('No MCTNet tasks found.')
        return
    for t in mctnet_tasks:
        status = t.status()
        print(f"{status['description']:45s} | {status['state']:12s} | {status.get('error_message', '')}")

check_tasks()


In [ ]:

import time

def wait_for_tasks(poll_seconds=60):
    while True:
        tasks  = ee.batch.Task.list()
        active = [t for t in tasks
                  if 'MCTNet' in t.config.get('description', '')
                  and t.status()['state'] in ('READY', 'RUNNING')]
        if not active:
            print('All MCTNet tasks finished')
            check_tasks()
            break
        print(f'  {len(active)} task(s) still running... checking again in {poll_seconds}s')
        time.sleep(poll_seconds)





In [ ]:
import pandas as pd
import numpy as np
import glob, os, re

RAW_DIR    = './data/raw/'
OUTPUT_DIR = './data/processed/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

STATES  = ['Arkansas', 'California']
N_STEPS = 36
BANDS   = ['Blue','Green','Red','RE1','RE2','RE3','RE4','NIR','SWIR1','SWIR2']

print('Preprocessing config loaded')


In [ ]:
def preprocess(state):
    print(f'\n{"="*50}')
    print(f'Processing {state}')
    print('='*50)


    files = sorted(glob.glob(os.path.join(RAW_DIR, f'MCTNet_{state}_{YEAR}*.csv')))
    if not files:
        raise FileNotFoundError(f'No CSV found for {state} in {RAW_DIR}')
    df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    print(f'Loaded {len(files)} file(s) → {len(df)} rows × {len(df.columns)} cols')


    expected = [f'{b}_t{t:02d}' for b in BANDS for t in range(1, N_STEPS+1)]
    missing  = [c for c in expected if c not in df.columns]
    if missing:
        print(f'  ⚠ WARNING: {len(missing)} band columns missing — check GEE export')
    else:
        print(f'  All {len(expected)} band×time columns present')


    for band in BANDS:
        cols = [f'{band}_t{t:02d}' for t in range(1, N_STEPS+1)]
        present = [c for c in cols if c in df.columns]
        df[present] = df[present].ffill(axis=1).bfill(axis=1)


    for band in BANDS:
        cols = [f'{band}_t{t:02d}' for t in range(1, N_STEPS+1) if f'{band}_t{t:02d}' in df.columns]
        mx = df[cols].max().max()
        if mx > 1.5:
            print(f'  ⚠ WARNING: {band} max = {mx:.3f} (expected ≤ 1.0, check divide by 10000)')


    print('  Label distribution:')
    print('  ' + df['label'].value_counts().sort_index().to_string().replace('\n', '\n  '))


    out = os.path.join(OUTPUT_DIR, f'test_MCTNet_{state}_{YEAR}_fixed.csv')
    df.to_csv(out, index=False)
    print(f'  Saved → {out}')
    return df

for state in STATES:
    preprocess(state)

print('\nDone   ')


In [ ]:

for state in STATES:
    path = os.path.join(OUTPUT_DIR, f'test_MCTNet_{state}_{YEAR}_fixed.csv')
    df   = pd.read_csv(path, nrows=3)
    print(f'{state}: {df.shape[1]} columns')
    key_cols = ['label', '.geo', 'Blue_t01', 'Red_t18', 'SWIR2_t36']
    present  = [c for c in key_cols if c in df.columns]
    print(df[present].to_string())
    print()
